# All DSPy teleprompters on HotPotQA — Colab + OpenAI (self-contained)

Runs every major DSPy teleprompter on the same RAG program over HotPotQA, with a **local Chroma retriever** built from HotPotQA's own context passages — no dependency on DSPy's external ColBERTv2 endpoint (which is often down).

## Teleprompters covered

| # | Teleprompter | Optimises | Cost (rough) |
|---|---|---|---|
| 1 | LabeledFewShot | Demos from labelled trainset | Free at compile |
| 2 | BootstrapFewShot | Demos from successful traces | Cheap |
| 3 | BootstrapFewShotWithRandomSearch | Demos + random search | 5–10× more |
| 4 | KNNFewShot | Demos picked per-input via embeddings | Embed cost |
| 5 | COPRO | Instructions only | ~150 LM calls |
| 6 | MIPROv2 | Joint instruction × demo Bayesian search | ~1000 LM calls |
| 7 | Ensemble | Wraps multiple compiled programs | Inference-time |
| 8 | BootstrapFinetune | Fine-tunes weights | Fine-tune job |

Total cost on `gpt-4o-mini`: ~$1–3 if you run all sections.

## Cell 1 — Install

In [3]:
!pip install -q dspy-ai chromadb sentence-transformers

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.2/23.2 MB 57.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 16.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 331.0/331.0 kB 17.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 146.5/146.5 kB 9.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 64.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 62.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.1/72.1 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 180.2/180.2 kB 12.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.0/69.0 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 231.6/231.6 kB 15.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.

## Cell 2 — OpenAI key

Defensive: works whether or not Colab Secrets is configured.

In [1]:
import os
try:
    from google.colab import userdata
    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
except Exception:
    import getpass
    os.environ["OPENAI_API_KEY"] = getpass.getpass("Paste OpenAI key: ")
assert os.environ["OPENAI_API_KEY"].startswith("sk-")
print("Key loaded.")

Key loaded.


## Cell 3 — Configure DSPy LM (retriever set up later)

In [4]:
import dspy
lm = dspy.LM("openai/gpt-4o-mini", max_tokens=300, temperature=0)
dspy.configure(lm=lm)
print("LM configured: gpt-4o-mini")

LM configured: gpt-4o-mini


## Cell 4 — Load HotPotQA

In [5]:
from dspy.datasets import HotPotQA

dataset = HotPotQA(train_seed=1, train_size=20, eval_seed=2023,
                   dev_size=50, test_size=0)
trainset = [x.with_inputs("question") for x in dataset.train]
devset = [x.with_inputs("question") for x in dataset.dev]
print(f"Train: {len(trainset)}  Dev: {len(devset)}")
print(f"Sample Q: {trainset[0].question}")
print(f"Sample A: {trainset[0].answer}")
# Inspect what attrs are on each example so we know what to pull passages from
print(f"Example attrs: {list(vars(trainset[0]))}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

fullwiki/train-00000-of-00002.parquet:   0%|          | 0.00/166M [00:00<?, ?B/s]

fullwiki/train-00001-of-00002.parquet:   0%|          | 0.00/166M [00:00<?, ?B/s]

fullwiki/validation-00000-of-00001.parqu(…):   0%|          | 0.00/28.0M [00:00<?, ?B/s]

fullwiki/test-00000-of-00001.parquet:   0%|          | 0.00/27.6M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/90447 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/7405 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/7405 [00:00<?, ? examples/s]

Train: 20  Dev: 50
Sample Q: At My Window was released by which American singer-songwriter?
Sample A: John Townes Van Zandt
Example attrs: ['_store', '_demos', '_input_keys']


## Cell 5 — Build a LOCAL retriever

Uses a small Wikipedia-derived corpus (`rag-mini-wikipedia`) so the retriever has substance even if HotPotQA examples don't expose their context. Embedded with `all-MiniLM-L6-v2` (384-dim, fast, no API key needed) into in-memory Chroma.

In [6]:
!pip install -q datasets

import chromadb
from chromadb.config import Settings
from sentence_transformers import SentenceTransformer
from datasets import load_dataset

# Pull a compact Wikipedia corpus (~3000 passages — enough to demonstrate retrieval)
wiki = load_dataset("rag-datasets/rag-mini-wikipedia", "text-corpus", split="passages")
passages = [row["passage"] for row in wiki]
print(f"Loaded {len(passages)} Wikipedia passages")

# Embed locally
embedder = SentenceTransformer("all-MiniLM-L6-v2")

client = chromadb.Client(Settings(anonymized_telemetry=False))
try:
    client.delete_collection("hotpot")
except Exception:
    pass
collection = client.create_collection("hotpot")

BATCH = 128
for i in range(0, len(passages), BATCH):
    chunk = passages[i:i + BATCH]
    embs = embedder.encode(chunk, show_progress_bar=False).tolist()
    collection.add(
        ids=[f"p{i + j}" for j in range(len(chunk))],
        documents=chunk,
        embeddings=embs,
    )
print(f"Indexed {collection.count()} passages into Chroma")

README.md:   0%|          | 0.00/719 [00:00<?, ?B/s]

data/passages.parquet/part.0.parquet:   0%|          | 0.00/797k [00:00<?, ?B/s]

Generating passages split:   0%|          | 0/3200 [00:00<?, ? examples/s]

Loaded 3200 Wikipedia passages


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

... (8 outputs left out)

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Indexed 3200 passages into Chroma


## Cell 6 — Wrap the local index as a DSPy retriever and register it

In [9]:
import dspy
from dspy.dsp.utils import dotdict   # gives us attribute access

class LocalRetriever(dspy.Retrieve):
    def __init__(self, k=3):
        super().__init__(k=k)

    def forward(self, query_or_queries, k=None):
        k = k or self.k
        queries = [query_or_queries] if isinstance(query_or_queries, str) else list(query_or_queries)
        all_passages = []
        for q in queries:
            emb = embedder.encode(q).tolist()
            res = collection.query(query_embeddings=[emb], n_results=k)
            for doc in res["documents"][0]:
                all_passages.append(dotdict({"long_text": doc}))
        return all_passages       # return list, NOT a Prediction

dspy.configure(rm=LocalRetriever(k=3))

# Smoke test
test = dspy.Retrieve(k=2)("Who wrote Pride and Prejudice?")
print("Smoke test top-2 passages:")
for i, p in enumerate(test.passages, 1):
    print(f"  [{i}] {p[:120]}...")

Smoke test top-2 passages:
  [1] * Mosier, John., "Grant", Palgrave MacMillan, 2006 ISBN 1-4039-7136-6....
  [2] * Lewis, Lloyd, Captain Sam Grant, Little, Brown, and Co., 1950, ISBN 0-316-52348-8. ...


## Cell 7 — Define the RAG program

In [10]:
class GenerateAnswer(dspy.Signature):
    """Answer the question using the retrieved context. 1-5 word factoid answer."""
    context = dspy.InputField(desc="relevant passages")
    question = dspy.InputField()
    answer = dspy.OutputField(desc="1-5 words")

class RAG(dspy.Module):
    def __init__(self, num_passages=3):
        super().__init__()
        self.retrieve = dspy.Retrieve(k=num_passages)
        self.generate_answer = dspy.ChainOfThought(GenerateAnswer)

    def forward(self, question):
        ctx = self.retrieve(question).passages
        return self.generate_answer(context=ctx, question=question)

smoke = RAG()(question=trainset[0].question)
print(f"Q: {trainset[0].question}")
print(f"A: {smoke.answer}  (gold: {trainset[0].answer})")

Q: At My Window was released by which American singer-songwriter?
A: Not found in context  (gold: John Townes Van Zandt)


## Cell 8 — Metric + evaluator

In [11]:
import re, string

def normalize(s):
    s = s.lower()
    s = "".join(ch for ch in s if ch not in string.punctuation)
    s = re.sub(r"\b(a|an|the)\b", " ", s)
    return " ".join(s.split())

def exact_match(example, pred, trace=None):
    return normalize(example.answer) == normalize(pred.answer)

from dspy.evaluate import Evaluate
evaluator = Evaluate(devset=devset, metric=exact_match,
                     num_threads=4, display_progress=True, display_table=0)

results = {}

## Cell 9 — Section flags

Cheap by default. Flip the expensive ones on once you've confirmed the cheap path runs.

In [12]:
RUN_BASELINE          = True
RUN_LABELED_FEWSHOT   = True
RUN_BOOTSTRAP_FEWSHOT = True
RUN_BOOTSTRAP_RS      = True
RUN_KNN_FEWSHOT       = True
RUN_COPRO             = False
RUN_MIPROV2           = False
RUN_ENSEMBLE          = False
RUN_BOOTSTRAP_FT      = False

## Cell 10 — Baseline (no optimisation)

In [14]:
if RUN_BASELINE:
    result = evaluator(RAG())
    score = result.score if hasattr(result, "score") else float(result)
    results["Baseline"] = score
    print(f"Baseline EM: {score:.1f}")

Average Metric: 2.00 / 50 (4.0%): 100%|██████████| 50/50 [00:05<00:00,  9.47it/s]

2026/05/03 15:25:13 INFO dspy.evaluate.evaluate: Average Metric: 2 / 50 (4.0%)



Baseline EM: 4.0


In [17]:
def run_eval(program):
    """Wrap evaluator() to handle both old (float) and new (EvaluationResult) return types."""
    r = evaluator(program)
    return r.score if hasattr(r, "score") else float(r)

## Cell 11 — LabeledFewShot

Picks k labelled examples from the trainset and inserts them as demos. **No LM calls during compile** — just demo selection. Fastest possible improvement over baseline.

In [18]:
if RUN_LABELED_FEWSHOT:
    from dspy.teleprompt import LabeledFewShot
    compiled = LabeledFewShot(k=4).compile(RAG(), trainset=trainset)
    results["LabeledFewShot"] = run_eval(compiled)
    print(f"LabeledFewShot EM: {results['LabeledFewShot']:.1f}")

Average Metric: 8.00 / 50 (16.0%): 100%|██████████| 50/50 [00:04<00:00, 12.05it/s]

2026/05/03 15:26:47 INFO dspy.evaluate.evaluate: Average Metric: 8 / 50 (16.0%)



LabeledFewShot EM: 16.0


## Cell 12 — BootstrapFewShot

Runs the program over the trainset, keeps traces where the metric passes, turns those into few-shot demos with reasoning chains baked in. Better than LabeledFewShot because the demos include the model's own reasoning.

In [19]:
if RUN_BOOTSTRAP_FEWSHOT:
    from dspy.teleprompt import BootstrapFewShot
    optimiser = BootstrapFewShot(metric=exact_match,
                                 max_bootstrapped_demos=4,
                                 max_labeled_demos=4)
    bfs_compiled = optimiser.compile(RAG(), trainset=trainset)
    results["BootstrapFewShot"] = run_eval(bfs_compiled)
    print(f"BootstrapFewShot EM: {results['BootstrapFewShot']:.1f}")

100%|██████████| 20/20 [01:30<00:00,  4.54s/it]


Bootstrapped 3 full traces after 19 examples for up to 1 rounds, amounting to 20 attempts.
Average Metric: 13.00 / 50 (26.0%): 100%|██████████| 50/50 [00:24<00:00,  2.06it/s]

2026/05/03 15:28:56 INFO dspy.evaluate.evaluate: Average Metric: 13 / 50 (26.0%)



BootstrapFewShot EM: 26.0


## Cell 13 — BootstrapFewShotWithRandomSearch

Generates many candidate programs (different random demo subsets), evaluates each on a held-out valset, picks the best. Often the best price/perf optimiser.

In [20]:
if RUN_BOOTSTRAP_RS:
    from dspy.teleprompt import BootstrapFewShotWithRandomSearch
    optimiser = BootstrapFewShotWithRandomSearch(
        metric=exact_match,
        max_bootstrapped_demos=4,
        max_labeled_demos=4,
        num_candidate_programs=8,
        num_threads=4,
    )
    rs_compiled = optimiser.compile(RAG(), trainset=trainset, valset=devset)
    results["BootstrapFewShotWithRandomSearch"] = run_eval(rs_compiled)
    print(f"BootstrapFewShotWithRandomSearch EM: {results['BootstrapFewShotWithRandomSearch']:.1f}")

Going to sample between 1 and 4 traces per predictor.
Will attempt to bootstrap 8 candidate sets.
Average Metric: 2.00 / 50 (4.0%): 100%|██████████| 50/50 [00:04<00:00, 12.12it/s]

2026/05/03 15:29:00 INFO dspy.evaluate.evaluate: Average Metric: 2 / 50 (4.0%)



New best score: 4.0 for seed -3
Scores so far: [4.0]
Best score so far: 4.0
Average Metric: 8.00 / 50 (16.0%): 100%|██████████| 50/50 [00:03<00:00, 14.02it/s]

2026/05/03 15:29:03 INFO dspy.evaluate.evaluate: Average Metric: 8 / 50 (16.0%)



New best score: 16.0 for seed -2
Scores so far: [4.0, 16.0]
Best score so far: 16.0


100%|██████████| 20/20 [00:09<00:00,  2.20it/s]


... (33 outputs left out)

2026/05/03 15:37:29 INFO dspy.evaluate.evaluate: Average Metric: 12 / 50 (24.0%)



Scores so far: [4.0, 16.0, 26.0, 26.0, 32.0, 24.0, 32.0, 28.0, 28.0, 24.0, 24.0]
Best score so far: 32.0
11 candidate programs found.
Average Metric: 16.00 / 50 (32.0%): 100%|██████████| 50/50 [00:04<00:00, 10.55it/s]

2026/05/03 15:37:34 INFO dspy.evaluate.evaluate: Average Metric: 16 / 50 (32.0%)



BootstrapFewShotWithRandomSearch EM: 32.0


## Cell 14 — KNNFewShot

Picks demos *per-input* via embedding nearest-neighbour against the trainset. Useful when inputs vary widely.

In [21]:
if RUN_KNN_FEWSHOT:
    from dspy.teleprompt import KNNFewShot
    from dspy import Embedder
    knn_embedder = Embedder("openai/text-embedding-3-small")
    optimiser = KNNFewShot(k=3, trainset=trainset, vectorizer=knn_embedder)
    compiled = optimiser.compile(RAG(), trainset=trainset)
    results["KNNFewShot"] = run_eval(compiled)
    print(f"KNNFewShot EM: {results['KNNFewShot']:.1f}")

TypeError: KNNFewShot.compile() got an unexpected keyword argument 'trainset'

## Cell 15 — COPRO

Coordinate-ascent over instructions only (no demos). Asks an LM to propose new instructions, evaluates each, keeps the best, iterates.

In [ ]:
if RUN_COPRO:
    from dspy.teleprompt import COPRO
    optimiser = COPRO(metric=exact_match, breadth=5, depth=2,
                      init_temperature=1.0)
    compiled = optimiser.compile(
        RAG(),
        trainset=trainset,
        eval_kwargs=dict(num_threads=4, display_progress=True, display_table=0),
    )
    score = evaluator(compiled)
    results["COPRO"] = score
    print(f"COPRO EM: {score:.1f}")

## Cell 16 — MIPROv2

Bayesian joint search over instructions × demos. DSPy's flagship optimiser. `auto="light"` is sensible for ~50 dev questions.

In [ ]:
if RUN_MIPROV2:
    from dspy.teleprompt import MIPROv2
    optimiser = MIPROv2(metric=exact_match, auto="light", num_threads=4)
    compiled = optimiser.compile(
        RAG(),
        trainset=trainset,
        valset=devset,
        requires_permission_to_run=False,
    )
    score = evaluator(compiled)
    results["MIPROv2"] = score
    print(f"MIPROv2 EM: {score:.1f}")

## Cell 17 — Ensemble (majority vote)

Wraps multiple compiled programs and combines their answers via majority vote. Requires `RUN_BOOTSTRAP_RS=True` so we have multiple candidates to ensemble.

In [ ]:
if RUN_ENSEMBLE:
    from dspy.teleprompt import BootstrapFewShotWithRandomSearch
    from dspy.teleprompt.ensemble import Ensemble
    from collections import Counter

    rs = BootstrapFewShotWithRandomSearch(
        metric=exact_match, max_bootstrapped_demos=4, max_labeled_demos=4,
        num_candidate_programs=4, num_threads=4,
    )
    rs.compile(RAG(), trainset=trainset, valset=devset)
    top_programs = [c["program"] for c in rs.candidate_programs[:3]]

    def majority_vote(preds):
        return Counter(p.answer for p in preds).most_common(1)[0][0]

    ensembled = Ensemble(reduce_fn=majority_vote).compile(top_programs)
    score = evaluator(ensembled)
    results["Ensemble"] = score
    print(f"Ensemble EM: {score:.1f}")

## Cell 18 — BootstrapFinetune

Distils the program into fine-tuned weights instead of optimising prompts. Creates an OpenAI fine-tune job (takes 5–30 minutes). Off by default.

In [ ]:
if RUN_BOOTSTRAP_FT:
    from dspy.teleprompt import BootstrapFinetune
    optimiser = BootstrapFinetune(metric=exact_match)
    compiled = optimiser.compile(
        RAG(),
        trainset=trainset,
        target="openai/gpt-4o-mini-2024-07-18",
    )
    score = evaluator(compiled)
    results["BootstrapFinetune"] = score
    print(f"BootstrapFinetune EM: {score:.1f}")

## Cell 19 — Compare all results

In [ ]:
import pandas as pd
df = pd.DataFrame(
    [{"Teleprompter": k, "EM (%)": v} for k, v in results.items()]
).sort_values("EM (%)", ascending=False)
print(df.to_string(index=False))

if "Baseline" in results:
    best = df['EM (%)'].max()
    print(f"\nBest lift over baseline: +{best - results['Baseline']:.1f} percentage points")

## Cell 20 — Inspect what BootstrapFewShot baked in

In [ ]:
if RUN_BOOTSTRAP_FEWSHOT:
    print("Instructions on the answer Predict:")
    print(" ", bfs_compiled.generate_answer.signature.instructions)
    print(f"\nDemos baked in: {len(bfs_compiled.generate_answer.demos)}")
    for i, d in enumerate(bfs_compiled.generate_answer.demos[:1], 1):
        print(f"\n--- Demo {i} ---")
        print(f"Q: {d.question}")
        if hasattr(d, 'reasoning'):
            print(f"Reasoning: {d.reasoning[:200]}...")
        print(f"A: {d.answer}")

## Cell 21 — Save and reload a compiled program

In [ ]:
if RUN_BOOTSTRAP_FEWSHOT:
    bfs_compiled.save("compiled_rag.json")
    fresh = RAG()
    fresh.load("compiled_rag.json")
    out = fresh(question="Who wrote Pride and Prejudice?")
    print(f"Reloaded program answer: {out.answer}")